# Module 1 - Deep Learning voi TensorFlow/Keras

Notebook nay gom **5 phan**, moi phan la mot bai hoc doc lap ve mot chu de nen tang cua Deep
Learning, tat ca dung chung mot bo du lieu y te that (**Pima Indians Diabetes**) va dung
`tf.keras` de thuc hanh:

1. **Xay dung va huan luyen mang DNN** - kien truc `Dense`, forward/backward/gradient descent.
2. **Tinh dao ham tu dong (Autodiff) va Gradient Checking** - `tf.GradientTape`.
3. **Khoi tao trong so va Vanishing/Exploding Gradient** - `kernel_initializer`.
4. **Cac thuat toan toi uu hoa** - GD, Momentum, Adam.
5. **Batch Normalization va Layer Normalization** - `BatchNormalization`, `LayerNormalization`.

Moi phan deu co ly thuyet, bai tap (dien vao cho trong, co dap an dang form thu gon cua Colab),
va phan ket qua/phan tich rieng.

## Cai dat & tai du lieu

Chi can 1 file CSV duy nhat (**khong can clone ca repo**) - cell duoi day tu tai ve neu chua co.

In [ ]:
import os
import urllib.request

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # tat toi uu oneDNN de dam bao ket qua giong het nhau
# giua cac lan chay - phai dat TRUOC khi 'import tensorflow' o cell sau (oneDNN co the doi thu
# tu tinh toan tren nhieu luong CPU, gay lech ket qua nho o vai chu so thap phan cuoi giua cac lan)

DATA_URL = (
    "https://raw.githubusercontent.com/NonomiyaIzumi/AI-Teaching-Labs/master/"
    "Deep%20Learning/Module%201/data/pima-indians-diabetes.csv"
)
DATA_PATH = "pima-indians-diabetes.csv"

if not os.path.isfile(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Da tai du lieu ve:", DATA_PATH)
else:
    print("Du lieu da co san:", DATA_PATH)

## Du lieu dung chung: Pima Indians Diabetes

Bo du lieu y te thuc te gom **768 benh nhan nu goc An Do (Pima)**, moi nguoi co **8 dac trung lam
sang** va 1 nhan nhi phan (co/khong mac tieu duong trong vong 5 nam):

| Dac trung | Y nghia |
|---|---|
| Pregnancies | So lan mang thai |
| Glucose | Nong do glucose trong huyet tuong (do sau 2h uong dung dich glucose) |
| BloodPressure | Huyet ap tam truong (mm Hg) |
| SkinThickness | Do day nep gap da co tay (mm) |
| Insulin | Nong do insulin huyet thanh sau 2h (mu U/ml) |
| BMI | Chi so khoi co the |
| DiabetesPedigreeFunction | Chi so di truyen (tien su gia dinh mac tieu duong) |
| Age | Tuoi |
| **Outcome** | **Nhan: 0 = khong mac tieu duong, 1 = mac tieu duong** |

**Van de thuc te can xu ly:** 5 cot (`Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`)
khong the co gia tri **0** ve mat sinh ly - gia tri `0` o day thuc chat la du lieu bi thieu (khong
do duoc), khong phai gia tri that. Neu giu nguyen, mo hinh se "hoc" sai rang `0` la mot gia tri binh
thuong. Cach xu ly: **median imputation** - thay `0` bang trung vi (median) cua cot do, tinh **tren
tap train** roi ap dung lai cho tap test (tranh **data leakage** - de lo thong tin tu tap test vao
qua trinh xu ly).

Sau khi thay gia tri thieu, du lieu duoc **chuan hoa** (chuan hoa Z-score: tru trung binh, chia do
lech chuan - cung tinh tren tap train) truoc khi dua vao mang - giup gradient descent hoi tu on
dinh hon, vi cac dac trung nhu `Age` (hang chuc) va `DiabetesPedigreeFunction` (hang phan thap phan)
co thang do rat khac nhau.

Ham `load_dataset()` va 2 ham tien ich `compile_and_train()`/`evaluate_classification()` duoi day
duoc dung chung xuyen suot ca 5 phan cua notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
import sklearn.model_selection
import sklearn.metrics

tf.config.experimental.enable_op_determinism()  # dam bao ket qua giong het nhau giua cac lan chay


def load_dataset(csv_path=DATA_PATH, test_size=0.2, seed=1):
    '''
    Doc va tien xu ly bo du lieu Pima Indians Diabetes.

    Returns: train_X, train_Y (shape (8, m_train), (1, m_train)), test_X, test_Y (tuong tu voi
    m_test). Quy uoc (n_x, m) giong cac cong thuc ly thuyet - can .T truoc khi dua vao Keras
    (xem compile_and_train/evaluate_classification ben duoi).
    '''
    raw = np.genfromtxt(csv_path, delimiter=",", skip_header=1)
    X = raw[:, :8]
    y = raw[:, 8]

    X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )
    X_train = X_train.copy()
    X_test = X_test.copy()

    zero_as_missing_cols = [1, 2, 3, 4, 5]  # Glucose, BloodPressure, SkinThickness, Insulin, BMI
    for c in zero_as_missing_cols:
        col = X_train[:, c]
        median = np.median(col[col != 0])
        X_train[X_train[:, c] == 0, c] = median
        X_test[X_test[:, c] == 0, c] = median

    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    X_train = (X_train - mu) / sigma
    X_test = (X_test - mu) / sigma

    train_X, test_X = X_train.T, X_test.T
    train_Y = y_train.reshape(1, -1)
    test_Y = y_test.reshape(1, -1)
    return train_X, train_Y, test_X, test_Y


def compile_and_train(model, X_train, Y_train, optimizer, epochs=100, batch_size=None, verbose=0):
    '''Compile + fit; X_train/Y_train o quy uoc (n_x, m)/(1, m), can .T cho Keras (sample-first).'''
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    history = model.fit(X_train.T, Y_train.T, epochs=epochs, batch_size=batch_size, verbose=verbose)
    return history


def evaluate_classification(model, X, y, title):
    '''Du doan bang model.predict(), in Accuracy/Precision/Recall/F1 va ve confusion matrix + ROC.'''
    y_true = y.ravel().astype(int)
    y_prob = model.predict(X.T, verbose=0).ravel()
    y_pred = (y_prob > 0.5).astype(int)

    acc = (y_pred == y_true).mean()
    prec = sklearn.metrics.precision_score(y_true, y_pred, zero_division=0)
    rec = sklearn.metrics.recall_score(y_true, y_pred, zero_division=0)
    f1 = sklearn.metrics.f1_score(y_true, y_pred, zero_division=0)
    cm = sklearn.metrics.confusion_matrix(y_true, y_pred)

    print(title)
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}  (ty le phat hien dung ca mac benh - quan trong trong bai toan y te)")
    print(f"  F1-score:  {f1:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    axes[0].imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, str(cm[i, j]), ha="center", va="center",
                         color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
    axes[0].set_xticks([0, 1])
    axes[0].set_xticklabels(["No Diabetes", "Diabetes"])
    axes[0].set_yticks([0, 1])
    axes[0].set_yticklabels(["No Diabetes", "Diabetes"])
    axes[0].set_xlabel("Du doan")
    axes[0].set_ylabel("Thuc te")
    axes[0].set_title("Confusion Matrix\n" + title)

    fpr, tpr, _ = sklearn.metrics.roc_curve(y_true, y_prob)
    roc_auc = sklearn.metrics.auc(fpr, tpr)
    axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve\n" + title)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": roc_auc}


train_X, train_Y, test_X, test_Y = load_dataset()
print("train_X:", train_X.shape, " train_Y:", train_Y.shape)
print("test_X: ", test_X.shape, " test_Y: ", test_Y.shape)

---
# Phan 1: Xay dung va huan luyen mang DNN bang TensorFlow/Keras

**Muc tieu:**
- Hieu kien truc cua mot mang no-ron nhieu lop (Deep Neural Network - DNN): cac lop Dense, ham
  kich hoat (activation), vi sao mang can "sau" va vi sao can phi tuyen.
- Hieu ban chat cua 4 buoc huan luyen mot mang no-ron: **forward propagation**, **cost function**,
  **backward propagation**, **gradient descent** - va Keras tu dong hoa nhung buoc nay nhu the nao.
- Xay dung, huan luyen va danh gia mot mang DNN hoan chinh tren du lieu Pima Diabetes.

## 1.1. Mang no-ron nhieu lop (Deep Neural Network) la gi?

Mot mang DNN gom nhieu **lop** (layer) xep chong len nhau. Moi lop nhan dau vao tu lop truoc, ap
dung mot phep bien doi tuyen tinh roi mot ham phi tuyen, va truyen ket qua sang lop sau:

$$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}, \qquad A^{[l]} = g^{[l]}(Z^{[l]})$$

trong do `W^[l]`, `b^[l]` la trong so/bias cua lop `l`, va `g^[l]` la ham kich hoat (activation).
Voi bai toan **phan loai nhi phan** (0/1), kien truc pho bien la:

$$\text{Input} \;\to\; [\text{Dense} \to \text{ReLU}] \times (L-1) \;\to\; \text{Dense} \to \text{Sigmoid} \;\to\; \hat{y}$$

- **ReLU** (`g(z) = max(0, z)`) dung cho cac hidden layer - re, khong bi vanishing gradient nhu
  Sigmoid/Tanh khi `z` lon.
- **Sigmoid** (`g(z) = 1/(1+e^{-z})`) dung cho lop cuoi - nen dau ra ve khoang `(0, 1)`, doc duoc
  nhu mot xac suat `P(y=1 | x)`.
- **Vi sao can nhieu lop (sau) va can phi tuyen?** Neu bo activation phi tuyen, nhieu lop Dense
  chong len nhau chi tuong duong **mot** phep bien doi tuyen tinh duy nhat - mang se khong the hoc
  duoc cac quan he phuc tap, phi tuyen giua dac trung dau vao va nhan.

## 1.2. Bon buoc huan luyen mot mang no-ron

1. **Forward propagation** - dua du lieu `X` qua tung lop, thu duoc du doan `\hat{y} = A^{[L]}`.
2. **Cost function** - do "mang sai bao nhieu" bang **binary cross-entropy**:
   $$J = -\frac{1}{m}\sum_{i=1}^m \Big[y^{(i)}\log(\hat y^{(i)}) + (1-y^{(i)})\log(1-\hat y^{(i)})\Big]$$
3. **Backward propagation** - tinh dao ham `dJ/dW^[l]`, `dJ/db^[l]` cho MOI tham so bang chain rule.
4. **Cap nhat tham so (gradient descent)**:
   $$W^{[l]} := W^{[l]} - \alpha \, \frac{\partial J}{\partial W^{[l]}}$$

**Diem mau chot cua `tf.keras`:** ban van phai **tu thiet ke kien truc** (buoc 1), nhung buoc 2-4
duoc Keras tu dong hoa: `model.compile(loss=..., optimizer=...)` khai bao cost function va thuat
toan cap nhat; `model.fit(X, y, epochs=...)` tu chay forward, tu tinh gradient bang **automatic
differentiation** (chinh xac, khong xap xi), roi tu cap nhat tham so.

## 1.3. Xay dung kien truc mang bang `tf.keras.Sequential`

`keras.Sequential` cho phep khai bao mang bang cach liet ke tuan tu tung lop. Moi lop
`layers.Dense(units, activation=...)` tuong ung dung mot lop trong cong thuc muc 1.1 - Keras **tu
suy ra shape** cua `W^[l]`, `b^[l]` tu `units` cua lop truoc va lop hien tai.

Voi bai toan nay, ta dung kien truc `[8, 7, 1]`: 8 dac trung dau vao -> 1 hidden layer 7 neuron
(ReLU) -> 1 output neuron (Sigmoid).

### Bai tap - `build_model`

In [ ]:
def build_model(input_dim, hidden_units=7):
    '''
    Xay dung mang [input_dim, hidden_units, 1]: Dense(hidden_units, ReLU) -> Dense(1, Sigmoid).

    Returns: mot tf.keras.Model CHUA compile (chua co optimizer/loss).
    '''
    # (~4 dong) dung keras.Sequential([...]) voi keras.Input + layers.Dense
    # model = keras.Sequential([
    #     keras.Input(shape=(input_dim,)),
    #     layers.Dense(hidden_units, activation="relu"),
    #     layers.Dense(1, activation="sigmoid"),
    # ])
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE
    return model

In [ ]:
#@title Answer { display-mode: "form" }
def build_model(input_dim, hidden_units=7):
    '''
    Xay dung mang [input_dim, hidden_units, 1]: Dense(hidden_units, ReLU) -> Dense(1, Sigmoid).

    Returns: mot tf.keras.Model CHUA compile (chua co optimizer/loss).
    '''
    # (~4 dong) dung keras.Sequential([...]) voi keras.Input + layers.Dense
    # YOUR CODE STARTS HERE
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(hidden_units, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    # YOUR CODE ENDS HERE
    return model

In [ ]:
tf.random.set_seed(1)
init_1 = keras.initializers.GlorotUniform(seed=1)  # gan seed rieng de ket qua on dinh giua cac lan chay
model = build_model(train_X.shape[0], hidden_units=7)
model.summary()

`model.summary()` in ra so tham so cua tung lop - doi chieu voi cong thuc: lop `Dense(7)` dau tien
co `8*7 + 7 = 63` tham so (`W1` shape `(7,8)` + `b1` shape `(7,)`), lop `Dense(1)` co `7*1 + 1 = 8`
tham so.

## 1.4. Bien dich va huan luyen mo hinh

`model.compile(...)` khai bao 3 thanh phan: `loss="binary_crossentropy"` (cong thuc cost function
`J`), `optimizer=keras.optimizers.SGD(learning_rate=...)` (thuat toan cap nhat tham so), va
`metrics=["accuracy"]` (chi so theo doi them, khong dung de tinh gradient).

`model.fit(X, y, epochs=..., batch_size=...)` chay vong lap huan luyen: `batch_size=None` nghia la
**full-batch gradient descent** - moi epoch dung toan bo `m` vi du de tinh mot lan cap nhat duy
nhat. Ta huan luyen `3000` epoch voi `learning_rate=0.5`.

In [ ]:
history = compile_and_train(
    model, train_X, train_Y,
    optimizer=keras.optimizers.SGD(learning_rate=0.5),
    epochs=3000, batch_size=None, verbose=0,
)
print(f"Cost cuoi cung (train):     {history.history['loss'][-1]:.4f}")
print(f"Accuracy cuoi cung (train): {history.history['accuracy'][-1]:.4f}")

## 1.5. Danh gia mo hinh tren tap test

Voi bai toan y te, **chi nhin Accuracy la khong du**: neu ~65% benh nhan trong tap khong mac benh,
mot mo hinh "ngu" luon du doan "khong mac benh" cung dat Accuracy ~65% ma khong hoc duoc gi. Can
nhin them **Precision**, **Recall** (quan trong nhat trong y te - ty le phat hien dung ca that su
co benh), **F1-score**, **confusion matrix**, va **ROC curve/AUC**.

In [ ]:
results_1 = evaluate_classification(model, test_X, test_Y, "Phan 1: tf.keras Sequential [8, 7, 1] (test set)")

## 1.6. Ket qua & Phan tich

Voi kien truc `[8, 7, 1]`, `learning_rate=0.5`, `3000` epoch full-batch: mo hinh dat **train
Accuracy ~81%, test Accuracy ~75%** (cac con so co the lech nhau vai phan tram giua cac lan chay
du da co dinh seed - dac diem binh thuong cua gradient descent tren CPU nhieu luong). Mo hinh hoc
duoc tin hieu du doan that su (ro rang tot hon baseline ~65% neu doan toan bo ve lop da so), nhung
Recall con co the cai thien them - mot huong tot cho phan **Bai tap mo rong**.

**Diem mau chot:** kien truc mang, cost function, va thuat toan cap nhat tham so (Dense + ReLU/
Sigmoid, binary cross-entropy, gradient descent) la nhung khai niem toan hoc **can ban** cua moi
mang no-ron. Framework khong "hoc gioi hon" - no tu dong hoa phan tinh dao ham va vong lap cap
nhat, de ban tap trung vao **thiet ke kien truc va chon sieu tham so**.

## 1.7. Bai tap mo rong

1. **Tang do sau/do rong:** thu `hidden_units=20` hoac them mot lop `Dense` nua - Accuracy/Recall
   tren test set thay doi the nao?
2. **Doi optimizer:** thay `keras.optimizers.SGD` bang `keras.optimizers.Adam` - so sanh toc do
   hoi tu (Phan 4 se di sau vao chu de nay).
3. **Cai thien Recall:** thu `model.fit(..., class_weight={0: 1, 1: w})` voi `w > 1` de "phat" mo
   hinh nang hon khi bo sot ca mac benh.
4. **Ve duong cost:** dung `history.history["loss"]` de ve bieu do cost theo epoch.

---
# Phan 2: Tinh dao ham tu dong (Automatic Differentiation) va Gradient Checking

**Muc tieu:**
- Hieu **automatic differentiation (autodiff)** la gi, khac gi voi tinh dao ham bang tay (giai
  tich) va tinh dao ham xap xi (numerical/finite difference).
- Biet dung `tf.GradientTape` - cong cu cot loi cua TensorFlow de tinh gradient tu dong.
- Hieu ky thuat **gradient checking**: dung dao ham xap xi de KIEM CHUNG dao ham tinh boi autodiff.

## 2.1. Ba cach tinh dao ham

Khi huan luyen mang no-ron bang gradient descent, ta can dao ham `dJ/d\theta` cua ham mat mat `J`
theo tung tham so `\theta`. Co 3 cach de co duoc con so nay:

1. **Dao ham giai tich (analytic/symbolic)** - tu tay ap dung quy tac chuoi, suy ra cong thuc roi
   lap trinh dung cong thuc do. Nhanh va chinh xac **neu suy dung**, nhung de sai khi ham phuc tap.
2. **Dao ham xap xi (numerical/finite difference)** - khong can biet cong thuc:
   $$\frac{dJ}{d\theta} \approx \frac{J(\theta+\varepsilon) - J(\theta-\varepsilon)}{2\varepsilon}$$
   Luon "dung xap xi" nhung **cham** (phai goi lai `J` 2 lan cho MOI tham so) nen khong dung de
   huan luyen, chi dung de **kiem chung**.
3. **Dao ham tu dong (autodiff)** - framework **tu dong ghi lai** moi phep toan da thuc hien (mot
   "computation graph"), roi ap dung chain rule tu dong nguoc lai tu output ve input. Ket qua
   **chinh xac tuyet doi** va **nhanh** - day chinh la co che TensorFlow/PyTorch/Keras dung de
   huan luyen mang no-ron.

## 2.2. `tf.GradientTape`

`tf.GradientTape` la cong cu autodiff cua TensorFlow: no "ghi bang" moi phep toan thuc hien tren
cac `tf.Variable` ben trong khoi `with tf.GradientTape() as tape:`, sau do `tape.gradient(y, x)`
tra ve `dy/dx`.

### Bai tap - `dtheta_autodiff`

Vi du don gian: `J(theta) = theta * x`. Ve mat giai tich, `dJ/dtheta = x`.

In [ ]:
def dtheta_autodiff(theta_value, x):
    '''
    Dung tf.GradientTape de tinh dJ/dtheta cho J(theta) = theta * x tai theta = theta_value.

    Returns: dtheta (python float)
    '''
    theta = tf.Variable(theta_value, dtype=tf.float64)
    # (~3 dong)
    # with tf.GradientTape() as tape:
    #     J = ...
    # dtheta = tape.gradient(J, theta)
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE
    return float(dtheta)

In [ ]:
#@title Answer { display-mode: "form" }
def dtheta_autodiff(theta_value, x):
    '''
    Dung tf.GradientTape de tinh dJ/dtheta cho J(theta) = theta * x tai theta = theta_value.

    Returns: dtheta (python float)
    '''
    theta = tf.Variable(theta_value, dtype=tf.float64)
    # (~3 dong)
    # YOUR CODE STARTS HERE
    with tf.GradientTape() as tape:
        J = theta * x
    dtheta = tape.gradient(J, theta)
    # YOUR CODE ENDS HERE
    return float(dtheta)

In [ ]:
x, theta_value = 2.0, 4.0
dtheta = dtheta_autodiff(theta_value, x)
print(f"J(theta) = theta * x  tai theta={theta_value}, x={x}")
print(f"dtheta (tu GradientTape) = {dtheta}")
print(f"dtheta ly thuyet (= x)   = {x}")
assert abs(dtheta - x) < 1e-6, "GradientTape cho ket qua sai!"
print("\nDung - GradientTape tu suy ra dJ/dtheta = x ma khong can ai lap trinh cong thuc nay.")

## 2.3. Gradient checking cho mot mang no-ron

Autodiff luon dung **theo dinh nghia toan hoc** cua phep tinh ban da khai bao. Nhung neu ban tu
dinh nghia mot phep tinh moi va lo VIET SAI, autodiff se tinh dao ham **chinh xac cho cong thuc
sai** do - no khong tu phat hien ra ban dinh nghia sai. Day la luc **gradient checking** van con
gia tri, du dang dung framework hien dai.

Ta thu ap dung ky thuat nay cho mot mang no-ron nho `[8, 5, 3, 1]`, tren mot mini-batch 16 diem
lay tu du lieu Pima Diabetes da nap o phan Cai dat.

### Bai tap - `compute_gradients_autodiff`

In [ ]:
np.random.seed(1)
batch_idx = np.random.choice(train_X.shape[1], size=16, replace=False)
X_batch = train_X[:, batch_idx].T.astype("float32")  # (16, 8) - Keras quy uoc sample-first
Y_batch = train_Y[:, batch_idx].T.astype("float32")  # (16, 1)
print("X_batch:", X_batch.shape, " Y_batch:", Y_batch.shape)

In [ ]:
def compute_gradients_autodiff(model, X, Y):
    '''
    Dung tf.GradientTape de tinh gradient cua binary cross-entropy loss theo TOAN BO
    model.trainable_variables cua mot mang Keras.

    Arguments:
    model -- tf.keras.Model (Sequential) da co trong so
    X, Y  -- tensor du lieu vao / nhan, sample-first (shape (m, n_x) va (m, 1))

    Returns: (loss, grads) - grads la list, cung do dai/thu tu voi model.trainable_variables
    '''
    # (~4 dong)
    # with tf.GradientTape() as tape:
    #     Y_hat = ...
    #     loss = ...
    # grads = tape.gradient(loss, model.trainable_variables)
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE
    return loss, grads

In [ ]:
#@title Answer { display-mode: "form" }
def compute_gradients_autodiff(model, X, Y):
    '''
    Dung tf.GradientTape de tinh gradient cua binary cross-entropy loss theo TOAN BO
    model.trainable_variables cua mot mang Keras.

    Arguments:
    model -- tf.keras.Model (Sequential) da co trong so
    X, Y  -- tensor du lieu vao / nhan, sample-first (shape (m, n_x) va (m, 1))

    Returns: (loss, grads) - grads la list, cung do dai/thu tu voi model.trainable_variables
    '''
    # (~4 dong)
    # YOUR CODE STARTS HERE
    with tf.GradientTape() as tape:
        Y_hat = model(X, training=False)
        loss = tf.reduce_mean(keras.losses.binary_crossentropy(Y, Y_hat))
    grads = tape.gradient(loss, model.trainable_variables)
    # YOUR CODE ENDS HERE
    return loss, grads

Ham `numerical_gradient_check` duoi day thuc hien dung dinh nghia toan hoc o muc 2.1 (cach 2): voi
moi toa do duoc chon ngau nhien trong `model.trainable_variables`, no tinh
`(J(theta+eps) - J(theta-eps)) / (2*eps)`, roi gop toan bo cac gia tri lai thanh **mot** sai so
tuong doi duy nhat:

$$\text{difference} = \frac{\|\text{grad}_{\text{autodiff}} - \text{grad}_{\text{approx}}\|}{\|\text{grad}_{\text{autodiff}}\| + \|\text{grad}_{\text{approx}}\|}$$

Nguong doc ket qua: `difference < 1e-7` la rat tot, `1e-7`-`1e-5` can xem xet them, `> 1e-3` gan
nhu chac chan co loi trong cong thuc.

In [ ]:
def numerical_gradient_check(model, X, Y, grads, epsilon=1e-4, num_checks=30, seed=1):
    '''
    Kiem chung gradient (lay ngau nhien mot so toa do tu cac trainable_variables) bang sai phan
    huu han, gop lai thanh MOT sai so tuong doi duy nhat theo cong thuc norm o tren.
    '''
    rng = np.random.default_rng(seed)
    grad_approx_list, grad_autodiff_list = [], []
    for _ in range(num_checks):
        var_idx = int(rng.integers(0, len(model.trainable_variables)))
        var = model.trainable_variables[var_idx]
        flat_size = int(np.prod(var.shape))
        idx = np.unravel_index(int(rng.integers(0, flat_size)), var.shape)

        original = var.numpy()
        value0 = original[idx]

        perturbed = original.copy()
        perturbed[idx] = value0 + epsilon
        var.assign(perturbed)
        Y_plus = model(X, training=False)
        loss_plus = float(tf.reduce_mean(keras.losses.binary_crossentropy(Y, Y_plus)))

        perturbed[idx] = value0 - epsilon
        var.assign(perturbed)
        Y_minus = model(X, training=False)
        loss_minus = float(tf.reduce_mean(keras.losses.binary_crossentropy(Y, Y_minus)))

        var.assign(original)

        grad_approx_list.append((loss_plus - loss_minus) / (2 * epsilon))
        grad_autodiff_list.append(float(grads[var_idx].numpy()[idx]))

    grad_approx = np.array(grad_approx_list)
    grad_autodiff = np.array(grad_autodiff_list)
    difference = np.linalg.norm(grad_autodiff - grad_approx) / (
        np.linalg.norm(grad_autodiff) + np.linalg.norm(grad_approx)
    )
    return difference, grad_autodiff, grad_approx


tf.random.set_seed(4)
init_2 = keras.initializers.GlorotUniform(seed=4)
model = keras.Sequential([
    keras.Input(shape=(8,)),
    layers.Dense(5, activation="relu", kernel_initializer=init_2),
    layers.Dense(3, activation="relu", kernel_initializer=init_2),
    layers.Dense(1, activation="sigmoid", kernel_initializer=init_2),
])

loss, grads = compute_gradients_autodiff(model, tf.constant(X_batch), tf.constant(Y_batch))
print(f"Loss tren minibatch 16 diem: {loss:.4f}")

difference, grad_autodiff, grad_approx = numerical_gradient_check(
    model, tf.constant(X_batch), tf.constant(Y_batch), grads, num_checks=50,
)
print(f"\ndifference (50 toa do duoc lay mau, gop bang norm) = {difference:.2e}")
assert difference < 5e-3, "Gradient tu GradientTape khong khop voi numerical gradient!"
print("\nNho - GradientTape cho gradient CHINH XAC (autodiff, khong xap xi), gan nhu tuyet doi")
print("khop voi dao ham xap xi (numerical) tinh doc lap.")

## 2.4. Tong ket

- **Autodiff** (`tf.GradientTape`) tinh dao ham chinh xac bang chain rule tu dong, khong xap xi va
  khong doi hoi ban tu suy cong thuc.
- **Numerical gradient** van chinh xac ve mat toan hoc nhung qua cham de dung trong vong lap huan
  luyen - vai tro cua no la **cong cu kiem chung doc lap**.
- **Gradient checking** van con gia tri thuc te ngay ca khi dung framework: quan trong nhat khi ban
  **tu viet mot phep tinh/loss/layer moi** ma Keras chua co san (Phan 5 se gap lai y tuong nay).

---
# Phan 3: Khoi tao trong so va Vanishing/Exploding Gradient

**Muc tieu:**
- Hieu vi sao **cach khoi tao trong so ban dau** anh huong quyet dinh den viec mang no-ron co hoc
  duoc hay khong - du kien truc va du lieu giu nguyen.
- Hieu 2 hien tuong **vanishing gradient** va **exploding gradient**.
- Biet dung `kernel_initializer` cua `tf.keras.layers.Dense`, va hieu vi sao **He initialization**
  thuong la lua chon mac dinh tot cho mang dung ReLU.

## 3.1. Vi sao khoi tao trong so lai quan trong?

### Khoi tao bang 0 (`zeros`)

Neu **moi** trong so trong cung mot lop bat dau bang 0, moi neuron trong lop do se luon nhan duoc
CUNG mot gradient trong suot qua trinh huan luyen - cac neuron "dong bo" voi nhau mai mai (**van
de doi xung - symmetry**). Du co bao nhieu neuron trong mot lop, chung hoc y het nhau, tuong duong
nhu lop do chi co **dung 1 neuron**.

### Khoi tao ngau nhien nhung QUA LON (vd `randn(...) * 10`)

Pha vo duoc tinh doi xung, nhung neu phuong sai qua cao, gia tri `Z^[l]` o cac lop sau se rat lon.
Voi Sigmoid/Tanh, `|Z|` lon dua ham vao vung **bao hoa** (dao ham gan bang 0) - gradient lan truyen
nguoc qua nhieu lop se **nhan don dan lai** (vanishing gradient). Voi ReLU, gia tri qua lon/qua am
co the day mang vao trang thai bat on dinh, cost dao dong manh hoac "no" (`exploding gradient`).

### Khoi tao co kiem soat theo do sau mang (Xavier/Glorot, He)

Y tuong chung: chon **phuong sai** cua trong so ti le nghich voi so neuron dau vao cua lop
(`fan_in`) de giu on dinh phuong sai cua `Z^[l]` qua cac lop:

$$\text{Xavier/Glorot: } W^{[l]} \sim \mathcal{N}\!\left(0, \frac{1}{n^{[l-1]}}\right)
\qquad\qquad
\text{He: } W^{[l]} \sim \mathcal{N}\!\left(0, \frac{2}{n^{[l-1]}}\right)$$

**He initialization** (He et al., 2015) nhan them he so 2 - phu hop hon cho **ReLU**, vi ReLU "cat"
mot nua truc so, nen can phuong sai lon hon o dau vao de bu lai phan bi mat sau khi qua ReLU.

## 3.2. Khoi tao trong so trong `tf.keras`

Moi lop `layers.Dense(...)` co tham so `kernel_initializer` (cho `W`) va `bias_initializer` (cho
`b`, mac dinh la 0). Cac chien luoc o muc 3.1 deu co san:

| Chien luoc | Class Keras |
|---|---|
| Khoi tao bang 0 | `keras.initializers.Zeros()` |
| Ngau nhien, phuong sai lon | `keras.initializers.RandomNormal(stddev=10.0)` |
| He initialization | `keras.initializers.HeNormal()` |
| Xavier/Glorot initialization | `keras.initializers.GlorotNormal()` (**mac dinh cua `Dense`**) |

## 3.3. Xay dung mang voi initializer tuy chinh - Bai tap

De quan sat ro vanishing/exploding, ta dung mot mang du sau (`[8, 10, 5, 1]`, 2 hidden layer).

### Bai tap - `build_model_with_initializer`

In [ ]:
def build_model_with_initializer(input_dim, initializer):
    '''
    Xay dung kien truc [input_dim, 10, 5, 1] (Dense -> ReLU -> Dense -> ReLU -> Dense -> Sigmoid),
    ap dung CUNG MOT initializer cho ca 3 lop Dense.

    Arguments:
    input_dim   -- so dac trung dau vao (8 cho Pima Diabetes)
    initializer -- mot tf.keras.initializers.Initializer (Zeros / RandomNormal / HeNormal ...)

    Returns: mot tf.keras.Model CHUA compile.
    '''
    # (~6 dong) dung keras.Sequential + layers.Dense(..., kernel_initializer=initializer)
    # model = keras.Sequential([
    #     keras.Input(shape=(input_dim,)),
    #     layers.Dense(10, activation="relu", kernel_initializer=initializer),
    #     layers.Dense(5, activation="relu", kernel_initializer=initializer),
    #     layers.Dense(1, activation="sigmoid", kernel_initializer=initializer),
    # ])
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE
    return model

In [ ]:
#@title Answer { display-mode: "form" }
def build_model_with_initializer(input_dim, initializer):
    '''
    Xay dung kien truc [input_dim, 10, 5, 1] (Dense -> ReLU -> Dense -> ReLU -> Dense -> Sigmoid),
    ap dung CUNG MOT initializer cho ca 3 lop Dense.

    Arguments:
    input_dim   -- so dac trung dau vao (8 cho Pima Diabetes)
    initializer -- mot tf.keras.initializers.Initializer (Zeros / RandomNormal / HeNormal ...)

    Returns: mot tf.keras.Model CHUA compile.
    '''
    # (~6 dong) dung keras.Sequential + layers.Dense(..., kernel_initializer=initializer)
    # YOUR CODE STARTS HERE
    model = keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(10, activation="relu", kernel_initializer=initializer),
        layers.Dense(5, activation="relu", kernel_initializer=initializer),
        layers.Dense(1, activation="sigmoid", kernel_initializer=initializer),
    ])
    # YOUR CODE ENDS HERE
    return model

## 3.4. Huan luyen & so sanh 3 cach khoi tao

Ca 3 lan train dung chung `learning_rate=0.01`, full-batch gradient descent, chi khac o
`kernel_initializer`.

In [ ]:
initializers = {
    "zeros":      keras.initializers.Zeros(),
    "random_bad": keras.initializers.RandomNormal(mean=0.0, stddev=10.0, seed=3),
    "he":         keras.initializers.HeNormal(seed=3),
}

EPOCHS = 2000
histories_3 = {}
for name, init in initializers.items():
    tf.random.set_seed(3)
    model = build_model_with_initializer(train_X.shape[0], init)
    history = compile_and_train(
        model, train_X, train_Y,
        optimizer=keras.optimizers.SGD(learning_rate=0.01),
        epochs=EPOCHS, batch_size=None, verbose=0,
    )
    histories_3[name] = {"model": model, "history": history}
    final_cost = history.history["loss"][-1]
    print(f"{name:10s} -> cost cuoi cung (train) = {final_cost:.4f}")

In [ ]:
plt.figure(figsize=(7, 4.5))
for name in initializers:
    plt.plot(histories_3[name]["history"].history["loss"], label=name)
plt.xlabel("epoch")
plt.ylabel("cost")
plt.title("Phan 3: Cost qua qua trinh train - so sanh kernel_initializer")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for name in initializers:
    evaluate_classification(histories_3[name]["model"], test_X, test_Y, f'Phan 3: kernel_initializer = "{name}" (test set)')

## 3.5. Ket qua & Phan tich

Sau 2000 epoch (`learning_rate=0.01`), ket qua tren test set: **`zeros`** ~65% accuracy nhung
**Precision = Recall = 0** - mang khong hoc duoc gi, dung nhu du doan (van de doi xung). **`random_bad`**
(`stddev=10`) cung ~65% accuracy, Recall = 0 - cost co giam nhe nhung van khong tach duoc lop nao.
**`he`** dat accuracy va Recall cao han han - hoc duoc mot tin hieu du doan that su.

**Diem mau chot:** cung mot kien truc, cung mot du lieu, cung mot thuat toan toi uu - chi **mot**
tham so khac nhau (cach khoi tao trong so ban dau) da tao ra chenh lech tu "khong hoc duoc gi" den
"hoc duoc mot mo hinh co ich". Day la ly do cac framework hien dai (Keras, PyTorch) deu chon san
**He/Xavier lam mac dinh** cho cac lop Dense/Conv.

## 3.6. Bai tap mo rong

1. **Xavier/Glorot cho ReLU:** thu doi `he` thanh `keras.initializers.GlorotNormal(seed=3)` - ket
   qua co kem hon `HeNormal` khong?
2. **Mang sau hon:** tang kien truc thanh `[8, 20, 20, 20, 10, 5, 1]` (5 hidden layer) - `zeros`
   va `random_bad` co "hong" nang hon khong?
3. **Theo doi gradient flow:** dung `tf.GradientTape` (Phan 2) de tinh va ghi lai `\|\|dW^{[l]}\|\|`
   sau moi vai epoch cho ca 3 cach khoi tao, roi ve bieu do.
4. **Ket hop voi Phan 5:** so sanh cach tiep can "chon khoi tao tot" (bai nay) voi cach tiep can
   "chuan hoa lai `Z` sau moi lop trong luc train" (BatchNorm/LayerNorm, Phan 5).

---
# Phan 4: Cac thuat toan toi uu hoa - GD, Momentum, Adam

**Muc tieu:**
- Hieu **mini-batch gradient descent** - vi sao chia du lieu thanh cac lo (batch) nho.
- Hieu ban chat cua 3 thuat toan cap nhat trong so pho bien nhat: **Gradient Descent (GD)**,
  **Momentum**, va **Adam** - va vi sao Adam thuong hoi tu nhanh hon trong thuc te.
- Biet chon va cau hinh optimizer trong `tf.keras` bang `keras.optimizers`.

## 4.1. Mini-batch gradient descent

Voi tap du lieu co `m` vi du, co 3 cach chia du lieu cho moi lan cap nhat trong so:

- **Batch (full-batch) gradient descent:** dung **toan bo** `m` vi du cho moi lan cap nhat. On
  dinh nhung cham voi du lieu lon.
- **Stochastic gradient descent (SGD thuan):** dung **tung 1** vi du cho moi lan cap nhat. Nhanh
  nhung rat "on ao".
- **Mini-batch gradient descent:** chia du lieu thanh cac **lo nho** (vd 32, 64, 128), cap nhat
  mot lan sau moi lo - dung hoa toc do va do on dinh, lua chon pho bien nhat trong thuc te.

Trong `tf.keras`, `model.fit(X, y, batch_size=64, epochs=...)` tu dong xao tron du lieu truoc moi
epoch, chia thanh cac mini-batch, va cap nhat cho tung batch.

## 4.2. Ba thuat toan cap nhat trong so

### Gradient Descent (GD) thuan

$$W := W - \alpha \, dW$$

Nhuoc diem: de dao dong qua lai trong khong gian tham so co dang "thung lung hep, dai", lam cham
hoi tu.

### Gradient Descent + Momentum

Tich luy mot "trung binh dong" (exponentially weighted moving average) cua gradient qua cac buoc
truoc, giong mot vien bi co da lan xuong doc:

$$v_{dW} := \beta \, v_{dW} + (1-\beta)\, dW \qquad\qquad W := W - \alpha \, v_{dW}$$

### Adam (Adaptive Moment Estimation)

Ket hop **Momentum** (trung binh dong bac 1, `v`) va y tuong cua RMSProp - trung binh dong bac 2
cua binh phuong gradient (`s`), dung de tu dieu chinh learning rate rieng cho tung tham so:

$$v_{dW} := \beta_1 v_{dW} + (1-\beta_1) dW \qquad\qquad s_{dW} := \beta_2 s_{dW} + (1-\beta_2) dW^2$$
$$W := W - \alpha \, \frac{v_{dW}}{\sqrt{s_{dW}} + \varepsilon}$$

Adam thuong hoi tu nhanh hon va it can dieu chinh learning rate thu cong hon, nen la lua chon mac
dinh pho bien trong thuc te.

## 4.3. Kien truc dung chung cho phan nay

Mang `[8, 5, 2, 1]` (2 hidden layer, khoi tao He) - vi phan nay tap trung vao **optimizer**, khong
phai kien truc/khoi tao (da hoc o Phan 3).

In [ ]:
def build_model_opt(input_dim):
    '''Kien truc co dinh [input_dim, 5, 2, 1], khoi tao kieu He - dung rieng cho Phan 4.'''
    he_init = keras.initializers.HeNormal(seed=3)
    return keras.Sequential([
        keras.Input(shape=(input_dim,)),
        layers.Dense(5, activation="relu", kernel_initializer=he_init),
        layers.Dense(2, activation="relu", kernel_initializer=he_init),
        layers.Dense(1, activation="sigmoid", kernel_initializer=he_init),
    ])

## 4.4. Chon optimizer - Bai tap

`tf.keras.optimizers` da cung cap san ca 3 thuat toan o muc 4.2:

- `keras.optimizers.SGD(learning_rate=...)` - GD thuan.
- `keras.optimizers.SGD(learning_rate=..., momentum=0.9)` - them tham so `momentum`.
- `keras.optimizers.Adam(learning_rate=..., beta_1=0.9, beta_2=0.999)` - Adam day du.

### Bai tap - `get_optimizer`

In [ ]:
def get_optimizer(name, learning_rate=0.0007):
    '''
    Tra ve mot optimizer Keras tuong ung: "gd" | "momentum" | "adam".

    Returns: mot tf.keras.optimizers.Optimizer
    '''
    # (~6 dong) dung keras.optimizers.SGD (co/khong momentum) va keras.optimizers.Adam
    # if name == "gd":
    #     ...
    # elif name == "momentum":
    #     ...
    # elif name == "adam":
    #     ...
    # YOUR CODE STARTS HERE
    pass
    # YOUR CODE ENDS HERE

In [ ]:
#@title Answer { display-mode: "form" }
def get_optimizer(name, learning_rate=0.0007):
    '''
    Tra ve mot optimizer Keras tuong ung: "gd" | "momentum" | "adam".

    Returns: mot tf.keras.optimizers.Optimizer
    '''
    # (~6 dong) dung keras.optimizers.SGD (co/khong momentum) va keras.optimizers.Adam
    # YOUR CODE STARTS HERE
    if name == "gd":
        return keras.optimizers.SGD(learning_rate=learning_rate)
    elif name == "momentum":
        return keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    elif name == "adam":
        return keras.optimizers.Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.999)
    else:
        raise ValueError(f"Unknown optimizer name: {name}")
    # YOUR CODE ENDS HERE

## 4.5. Huan luyen & so sanh 3 optimizer

Dung cung `learning_rate=0.0007` va `batch_size=64` (mini-batch gradient descent) cho ca 3
optimizer.

In [ ]:
EPOCHS = 200
histories_4 = {}
for name in ["gd", "momentum", "adam"]:
    tf.random.set_seed(3)
    model = build_model_opt(train_X.shape[0])
    optimizer = get_optimizer(name, learning_rate=0.0007)
    history = compile_and_train(model, train_X, train_Y, optimizer, epochs=EPOCHS, batch_size=64, verbose=0)
    histories_4[name] = {"model": model, "history": history}
    print(f"{name:10s} -> cost cuoi cung (train) = {history.history['loss'][-1]:.4f}")

In [ ]:
plt.figure(figsize=(7, 4.5))
for name in ["gd", "momentum", "adam"]:
    plt.plot(histories_4[name]["history"].history["loss"], label=name)
plt.xlabel("epoch")
plt.ylabel("cost")
plt.title("Phan 4: Cost qua qua trinh train - so sanh optimizer")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for name in ["gd", "momentum", "adam"]:
    evaluate_classification(histories_4[name]["model"], test_X, test_Y, f'Phan 4: optimizer = "{name}" (test set)')

## 4.6. Ket qua & Phan tich

Sau 200 epoch (`learning_rate=0.0007`, `batch_size=64`): thu tu **Adam > Momentum > GD** ve ca
cost lan Recall the hien dung ly thuyet o muc 4.2 - Momentum tich luy da theo huong gradient nhat
quan nen nhanh hon GD thuan; Adam ket hop them dieu chinh learning rate thich nghi theo tung tham
so nen hoi tu nhanh nhat trong ca ba. Ca 3 optimizer deu con cach xa diem hoi tu hoan toan sau chi
200 epoch - voi learning rate co dinh nho nhu the nay, can nhieu epoch hon nua de GD/Momentum bat
kip Adam.

**Diem mau chot:** voi cung mot kien truc, cung mot du lieu, chi doi thuat toan cap nhat trong so
da tao ra chenh lech dang ke ve toc do hoi tu trong cung mot ngan sach epoch.

## 4.7. Bai tap mo rong

1. **Tang so epoch:** thu `epochs=2000` hoac `5000` cho `gd`/`momentum` - chung co dan bat kip
   `adam` khong?
2. **Tinh chinh learning rate rieng:** dung `get_optimizer(name, learning_rate=...)` de thu cac
   learning rate khac nhau cho tung optimizer.
3. **Hoc rate decay:** ket hop `keras.optimizers.schedules.ExponentialDecay` lam `learning_rate`
   truyen vao optimizer.
4. **Doi batch_size:** thu `batch_size=16` va `batch_size=256` cho `adam` - quan sat anh huong den
   do "muot" cua duong cost va toc do hoi tu.

---
# Phan 5: Batch Normalization va Layer Normalization

**Muc tieu:**
- Hieu **Batch Normalization (BatchNorm)** va **Layer Normalization (LayerNorm)** - hai ky thuat
  chuan hoa gia tri **trong luc huan luyen**.
- Phan biet ro **truc tinh thong ke** cua BatchNorm (theo tung vi du trong batch) va LayerNorm
  (theo tung feature trong 1 vi du).
- Biet chen `layers.BatchNormalization()`/`layers.LayerNormalization()` dung vi tri trong kien
  truc `tf.keras`, va quan sat hieu qua thuc te khi mang bi khoi tao kem.

## 5.1. Vi sao can chuan hoa NGAY TRONG luc train?

Phan 3 da thay: khoi tao trong so kem co the khien phan phoi cua `Z^[l]` "troi" ve vung qua lon
hoac qua nho ngay tu dau. Nhung ngay ca voi khoi tao tot, phan phoi cua `Z^[l]` van co the **thay
doi dan** trong qua trinh train khi trong so cac lop truoc duoc cap nhat (**internal covariate
shift**).

**Y tuong cua BatchNorm/LayerNorm:** thay vi chi chon khoi tao that can than **mot lan** (Phan 3),
ta **chu dong chuan hoa lai** `Z^[l]` **o moi buoc forward**, giu phan phoi on dinh xuyen suot qua
trinh train. Sau khi chuan hoa ve trung binh 0/phuong sai 1, mot cap tham so hoc duoc `\gamma,
\beta` (scale & shift) cho phep mang **tu quyet dinh** neu no van can mot phan phoi khac 0/1:

$$\hat{Z} = \frac{Z - \mu}{\sqrt{\sigma^2 + \varepsilon}} \qquad\qquad \tilde{Z} = \gamma \hat{Z} + \beta$$

## 5.2. Batch Normalization vs Layer Normalization

Khac biet duy nhat giua hai ky thuat la **truc tinh** `\mu, \sigma^2`:

| | **BatchNormalization** | **LayerNormalization** |
|---|---|---|
| Tinh thong ke tren | Tung **feature**, gop theo **cac vi du trong batch** | Tung **vi du**, gop theo **cac feature** |
| Phu thuoc batch size? | **Co** - batch qua nho lam thong ke khong on dinh | **Khong** - tinh doc lap tren tung vi du |
| Train vs Test | Can co che rieng: dung thong ke cua batch khi train, dung trung binh truot khi test | Giong het nhau o train va test |
| Thuong dung cho | CNN/MLP voi batch size du lon | RNN/Transformer, hoac khi batch size nho/thay doi |

BatchNorm can co che "train/test khac nhau" vi luc test co the chi co **1** vi du. Giai phap: trong
luc train, mang con duy tri mot **trung binh truot** (exponential moving average) cua `\mu,
\sigma^2` qua cac batch; luc test, dung truc tiep trung binh truot nay.

## 5.3. Kien truc thi nghiem

Ta co tinh dung mot kien truc **sau** (`[8, 20, 20, 20, 20, 1]`, 4 hidden layer) **voi khoi tao
trong so kem** (`RandomNormal(stddev=4.0)`) - dung dieu kien de vanishing/exploding gradient the
hien ro, roi quan sat BatchNorm/LayerNorm co giup mang hoc duoc khong.

## 5.4. Chen lop chuan hoa vao kien truc - Bai tap

Vi tri chen chuan: **giua** phep bien doi tuyen tinh (`Dense`, KHONG activation) **va** ham kich
hoat phi tuyen (`Activation("relu")`).

### Bai tap - `build_model_with_norm`

In [ ]:
def build_model_with_norm(input_dim, norm_type="none", bad_init_stddev=4.0):
    '''
    Xay dung mang [input_dim, 20, 20, 20, 20, 1] voi khoi tao trong so CO CHU Y kem
    (RandomNormal stddev=4.0), chen them lop chuan hoa (neu co) GIUA moi Dense va ReLU
    tuy `norm_type`.

    Arguments:
    norm_type -- "none" | "batchnorm" | "layernorm"

    Returns: mot tf.keras.Model CHUA compile.
    '''
    bad_init = keras.initializers.RandomNormal(mean=0.0, stddev=bad_init_stddev, seed=3)
    layer_list = [keras.Input(shape=(input_dim,))]
    for _ in range(4):
        layer_list.append(layers.Dense(20, kernel_initializer=bad_init))
        # (~4 dong) neu norm_type == "batchnorm": them layers.BatchNormalization()
        #           neu norm_type == "layernorm": them layers.LayerNormalization()
        #           neu norm_type == "none":      khong them gi
        #           sau do LUON them layers.Activation("relu")
        # YOUR CODE STARTS HERE
        pass
        # YOUR CODE ENDS HERE
    layer_list.append(layers.Dense(1, kernel_initializer=bad_init, activation="sigmoid"))
    model = keras.Sequential(layer_list)
    return model

In [ ]:
#@title Answer { display-mode: "form" }
def build_model_with_norm(input_dim, norm_type="none", bad_init_stddev=4.0):
    '''
    Xay dung mang [input_dim, 20, 20, 20, 20, 1] voi khoi tao trong so CO CHU Y kem
    (RandomNormal stddev=4.0), chen them lop chuan hoa (neu co) GIUA moi Dense va ReLU
    tuy `norm_type`.

    Arguments:
    norm_type -- "none" | "batchnorm" | "layernorm"

    Returns: mot tf.keras.Model CHUA compile.
    '''
    bad_init = keras.initializers.RandomNormal(mean=0.0, stddev=bad_init_stddev, seed=3)
    layer_list = [keras.Input(shape=(input_dim,))]
    for _ in range(4):
        layer_list.append(layers.Dense(20, kernel_initializer=bad_init))
        # (~4 dong)
        # YOUR CODE STARTS HERE
        if norm_type == "batchnorm":
            layer_list.append(layers.BatchNormalization())
        elif norm_type == "layernorm":
            layer_list.append(layers.LayerNormalization())
        layer_list.append(layers.Activation("relu"))
        # YOUR CODE ENDS HERE
    layer_list.append(layers.Dense(1, kernel_initializer=bad_init, activation="sigmoid"))
    model = keras.Sequential(layer_list)
    return model

## 5.5. Huan luyen & so sanh none / batchnorm / layernorm

Dung `learning_rate=0.1`, full-batch gradient descent cho ca 3 bien the.

In [ ]:
EPOCHS = 2000
histories_5 = {}
for norm_type in ["none", "batchnorm", "layernorm"]:
    tf.random.set_seed(3)
    model = build_model_with_norm(train_X.shape[0], norm_type=norm_type)
    history = compile_and_train(
        model, train_X, train_Y,
        optimizer=keras.optimizers.SGD(learning_rate=0.1),
        epochs=EPOCHS, batch_size=None, verbose=0,
    )
    histories_5[norm_type] = {"model": model, "history": history}
    print(f'{norm_type:10s} -> cost cuoi cung (train) = {history.history["loss"][-1]:.4f}')

In [ ]:
plt.figure(figsize=(7, 4.5))
for norm_type in ["none", "batchnorm", "layernorm"]:
    plt.plot(histories_5[norm_type]["history"].history["loss"], label=norm_type)
plt.xlabel("epoch")
plt.ylabel("cost")
plt.title("Phan 5: Cost qua qua trinh train - so sanh none / batchnorm / layernorm")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for norm_type in ["none", "batchnorm", "layernorm"]:
    evaluate_classification(histories_5[norm_type]["model"], test_X, test_Y, f'Phan 5: norm_type = "{norm_type}" (test set)')

## 5.6. Ket qua & Phan tich

Sau 2000 epoch (`learning_rate=0.1`): **`none`** cost dung yen o `~ln(2)`, test Accuracy ~65%
nhung **Precision = Recall = 0** - mang hoan toan khong hoc duoc gi voi khoi tao kem tren kien
truc sau nay. **`batchnorm`** va **`layernorm`** deu "cuu" duoc mang: cost train giam manh ve gan
0, test Accuracy len ~65-70% cho ca hai (cac con so cu the co the lech nhau vai phan tram giua cac
lan chay du da co dinh seed - mang cang sau/cang nhieu epoch thi cang co co hoi tich luy sai so
lam tron nho tren CPU nhieu luong, du ban chat dinh tinh luon nhat quan).

**Luu y ve overfitting:** cost train gan 0 trong khi test accuracy chi ~65-70% la dau hieu ro ret
cua **overfitting** - voi 4 hidden layer x 20 unit tren chi 614 vi du train, mang du suc "hoc
thuoc" toan bo tap train sau 2000 epoch full-batch. BatchNorm/LayerNorm giai quyet van de
vanishing/exploding gradient, nhung **khong** tu dong chong overfitting.

**Diem mau chot:** cung mot kien truc, cung mot khoi tao kem, chi them mot lop chuan hoa dung vi
tri da bien mot mang "chet" (Recall = 0) thanh mot mang hoc duoc tin hieu that su.

## 5.7. Bai tap mo rong

1. **Batch size = 1:** thu train `batchnorm` voi `batch_size=1` - du doan dieu gi se xay ra voi
   thong ke `\mu, \sigma^2` tinh tren batch chi co 1 diem? So sanh voi `layernorm`.
2. **Chuan hoa ca lop output:** thu them chuan hoa ngay truoc lop `Dense(1, activation="sigmoid")`
   cuoi cung - quan sat cost/accuracy co thay doi khong.
3. **Giam kien truc:** thu kien truc nong hon (1-2 hidden layer) voi cung khoi tao kem - `none` co
   con that bai hoan toan khong?
4. **So sanh voi khoi tao tot (Phan 3):** thay `bad_init_stddev=4.0` bang `keras.initializers.
   HeNormal()` cho ca 3 bien the - BatchNorm/LayerNorm co con tao ra khac biet lon nhu vay khong?

---
# Tong ket chung

Qua 5 phan, notebook nay di tu **kien truc co ban** (Phan 1) den **cong cu debug** (Phan 2 -
gradient checking/autodiff), roi den 3 yeu to quyet dinh mot mang no-ron co hoc duoc tot hay khong:
**khoi tao trong so** (Phan 3), **thuat toan toi uu** (Phan 4), va **chuan hoa trong luc train**
(Phan 5). Ca 5 phan dung chung mot bo du lieu that (Pima Indians Diabetes) va chung mot bo cong cu
danh gia (`evaluate_classification`), giup so sanh truc tiep hieu qua cua tung ky thuat tren cung
mot bai toan thuc te.

## Tai lieu tham khao

| Chu de | Lien ket |
|---|---|
| `tf.keras.Sequential` API | https://www.tensorflow.org/guide/keras/sequential_model |
| `tf.GradientTape` guide | https://www.tensorflow.org/guide/autodiff |
| He et al., 2015 - He initialization | https://arxiv.org/abs/1502.01852 |
| Glorot & Bengio, 2010 - Xavier initialization | https://proceedings.mlr.press/v9/glorot10a.html |
| Kingma & Ba, 2015 - Adam | https://arxiv.org/abs/1412.6980 |
| Ioffe & Szegedy, 2015 - Batch Normalization | https://arxiv.org/abs/1502.03167 |
| Ba, Kiros & Hinton, 2016 - Layer Normalization | https://arxiv.org/abs/1607.06450 |
| Pima Indians Diabetes Dataset (nguon goc) | https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database |